## Imports and configuration
Sets the session time zone to UTC (matches Kusto and Bronze), the job name, and two timestamp formats. The watermark text keeps microseconds so no row is reloaded.

In [ ]:
import json
import uuid
from datetime import datetime, timezone

import fleet_shared as fs
from pyspark.sql import functions as F, Window
from pyspark.sql.types import (StructType, StructField, StringType,
                               BooleanType, ArrayType)

spark.conf.set("spark.sql.session.timeZone", "UTC")   # match Kusto / Bronze timestamps

JOB_NAME = "nb_p2_batch_job_1"
TS_FMT = "%Y-%m-%d %H:%M:%S.%f"          # used inside SQL TIMESTAMP literals
WM_FMT = "%Y-%m-%dT%H:%M:%S.%fZ"         # watermark text keeps microseconds (no reloaded rows)

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 5, Finished, Available, Finished, False)

## Run manifest and job settings
Loads `Files/evidence/run_manifest.json` (`wall_start`, `sim_start`, acceleration factor A) for `lateness_sec`, and compares `run_summary.json` with the Bronze row counts so a manifest from a different run is noticed. Also holds the thresholds for the sequence flags, the vehicle-activity check and Gate 1.

**Notes:** the manifest describes one run. If Bronze contains several runs, `lateness_sec` is only right for the run in the manifest. The thresholds should be compared with the printed `fs.DEFAULT_RULES` of the library.

In [ ]:
# ---- Run manifest (needed for lateness_sec) and job settings ----
MANIFEST_PATH = "Files/evidence/run_manifest.json"     # must be the manifest of the run that is in Bronze
SUMMARY_PATH  = "Files/evidence/run_summary.json"

def _iso_to_epoch(s):
    return datetime.fromisoformat(s.replace("Z", "+00:00")).timestamp()

def load_manifest():
    m = spark.read.option("multiLine", "true").json(MANIFEST_PATH).first()
    return {"run_id": m["run_id"],
            "sim_start": _iso_to_epoch(m["sim_start"]),
            "wall_start": _iso_to_epoch(m["wall_start"]),
            "A": float(m["acceleration_factor"]),
            "fast_mode": bool(m["fast_mode"])}

def check_manifest_against_bronze():
    """Print whether the manifest/summary belong to the data in Bronze. Prints only, never stops the job."""
    s = spark.read.option("multiLine", "true").json(SUMMARY_PATH).first()
    per = s["per_signal"].asDict()
    ok = True
    for sig in ["gps", "engine", "fuel", "temperature"]:
        n = spark.table(f"bronze.bronze_{sig}_raw").count()
        same = (n == per[sig])
        ok = ok and same
        print(f"{sig:12s} summary={per[sig]:>7} bronze={n:>7} {'OK' if same else 'MISMATCH'}")
    print("run_id:", s["run_id"], "| messages_sent:", s["messages_sent"])
    if not ok:
        print("WARNING: the manifest does not match Bronze, lateness_sec would be wrong. Get the right files first.")
    if MANIFEST["fast_mode"]:
        print("WARNING: fast_mode = true, lateness_sec is meaningless for this run.")

MANIFEST = load_manifest()
print(MANIFEST)
check_manifest_against_bronze()

# ---- Settings for the sequence flags, vehicle check and Gate 1 ----
ENABLE_ACTIVE_CHECK = True      # reject rows whose vehicle is not active on the event date
SPIKE_DELTA_C       = 8.0       # temperature: jump from the previous reading above this -> SPIKE
FLATLINE_N          = 10        # temperature: this many identical readings in a row -> FLATLINE
MAX_JUMP_SPEED_KMH  = 200.0     # gps: implied speed between two consecutive points above this -> IMPOSSIBLE_JUMP
GATE_MAX_INVALID_PCT = 5.0      # quality_gate_max_invalid_pct

# Compare these thresholds with the library rules and edit the numbers above if they differ
print(fs.DEFAULT_RULES)

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 6, Finished, Available, Finished, False)

{'run_id': 'run-20261026-043040-smoke', 'sim_start': 1767248520.0, 'wall_start': 1792989040.0, 'A': 50.0, 'fast_mode': False}


gps          summary=  30895 bronze=  30895 OK


engine       summary=  30926 bronze=  30926 OK


fuel         summary=   5176 bronze=   5176 OK
temperature  summary=   7197 bronze=   7197 OK
run_id: run-20261026-043040-smoke | messages_sent: 74194
{'idle_speed_threshold_kmh': 2.0, 'idle_minutes_threshold': 5.0, 'speeding_tolerance_kmh': 5.0, 'gps_accuracy_max_meters': 50.0, 'deviation_consecutive_readings': 3.0, 'gps_jump_max_kmh': 200.0, 'speed_max_kmh': 130.0, 'engine_rpm_max': 4000.0, 'engine_temperature_max_c': 130.0, 'cargo_temperature_valid_min_c': -40.0, 'cargo_temperature_valid_max_c': 60.0, 'temperature_spike_c': 8.0, 'flatline_readings': 20.0, 'future_timestamp_tolerance_minutes': 5.0, 'expected_interval_gps_sec': 10.0, 'expected_interval_engine_sec': 10.0, 'expected_interval_fuel_sec': 60.0, 'expected_interval_temp_sec': 30.0, 'offline_missed_intervals': 3.0, 'late_data_watermark_minutes': 10.0, 'sequence_gap_grace_minutes': 120.0, 'dq_window_minutes': 10.0, 'invalid_rate_alert_pct': 2.0, 'quality_gate_max_invalid_pct': 5.0, 'score_weight_speeding': 35.0, 'score_weight_

## Window selection
Lower bound is the saved watermark in `ops.ops_watermark`; upper bound is the newest Bronze `ingestion_time`. Only rows in (lower, upper] are processed.

In [ ]:
def get_window(signal):
    """Return (lower, upper): lower = saved watermark, upper = newest Bronze ingestion_time."""
    lower = spark.sql(
        f"SELECT to_timestamp(watermark_utc) AS w FROM ops.ops_watermark WHERE signal='{signal}'"
    ).first()["w"]
    upper = (spark.table(f"bronze.bronze_{signal}_raw")
                  .agg(F.max("ingestion_time")).first()[0])
    return lower, upper

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 7, Finished, Available, Finished, False)

## Lookups and validation
`load_lookups` builds the known vehicles, known trips and tank capacity (latest row per vehicle by `updated_at`). `validate` runs `fleet_shared.validate_message` per row through a UDF and splits good and bad rows.

**Vehicle active check:** a valid row is also rejected when its vehicle has no `ref_vehicles_raw` version that is active on the event date (`active_from <= date <= active_to`, empty means open ended). Such rows go to the DLQ as `UNKNOWN_VEHICLE` with the message "Vehicle not active at event time". Switch it off with `ENABLE_ACTIVE_CHECK = False`.

**Pending:** the simulated `now` argument for the future-timestamp check.

In [ ]:
RES_SCHEMA = StructType([
    StructField("ok", BooleanType()),
    StructField("error_code", StringType()),
    StructField("error_message", StringType()),
    StructField("retryable", BooleanType()),
    StructField("flags", ArrayType(StringType())),
    StructField("record_json", StringType()),
])


def load_lookups():
    """Reference sets passed to fleet_shared so it can detect UNKNOWN_* and tank overflow."""
    veh = spark.table("ref.ref_vehicles_raw")
    known_vehicles = {r[0] for r in veh.select("vehicle_id").distinct().collect()}
    known_trips = {r[0] for r in spark.table("ref.ref_trips").select("trip_id").distinct().collect()}

    # One capacity per vehicle: take the latest row by updated_at (all columns are strings)
    w = Window.partitionBy("vehicle_id").orderBy(F.col("updated_at").desc())
    latest = (veh.withColumn("rn", F.row_number().over(w)).where("rn = 1")
                 .select("vehicle_id", F.col("fuel_tank_capacity_liters").cast("double").alias("cap")))
    tank_cap = {r["vehicle_id"]: r["cap"] for r in latest.collect() if r["cap"] is not None}
    return known_vehicles, known_trips, tank_cap


def make_validate_udf(known_vehicles, known_trips, tank_cap, now=None):
    """Build the UDF with the lookups captured in its closure (small sets, shipped to executors).
    `now` is the SIMULATED time (naive datetime) used by the library for the future-timestamp check."""
    @F.udf(RES_SCHEMA)
    def validate_udf(raw):
        r = fs.validate_message(raw, known_vehicles=known_vehicles,
                                known_trips=known_trips,
                                tank_capacity_by_vehicle=tank_cap,
                                now=now)
        return (r.ok, r.error_code, r.error_message, r.retryable,
                list(r.flags or []), json.dumps(r.record) if r.record else None)
    return validate_udf


def simulated_now(upper):
    """Simulated time at the newest Bronze ingestion_time: sim_start + (wall elapsed) * A.
    Returned as a naive datetime, because fleet_shared compares it with naive datetimes."""
    up_epoch = upper.replace(tzinfo=timezone.utc).timestamp()
    sim_epoch = MANIFEST["sim_start"] + (up_epoch - MANIFEST["wall_start"]) * MANIFEST["A"]
    return datetime.fromtimestamp(sim_epoch, tz=timezone.utc).replace(tzinfo=None)


def load_vehicle_intervals():
    """One row per vehicle version: the dates in which that version was active (NULL = open ended)."""
    return (spark.table("ref.ref_vehicles_raw")
                 .select(F.col("vehicle_id").alias("iv_vehicle"),
                         F.to_date(F.trim("active_from")).alias("iv_from"),
                         F.to_date(F.trim("active_to")).alias("iv_to")))


def split_by_vehicle_activity(good):
    """Split validated rows into (active, inactive). Inactive rows are returned in the DLQ shape
    (error_code UNKNOWN_VEHICLE, message says the vehicle was not active at event time)."""
    g = (good
         .withColumn("_veh", F.get_json_object("v.record_json", "$.vehicle_id"))
         .withColumn("_ev_date", F.to_date(F.get_json_object("v.record_json", "$.event_time").cast("timestamp"))))
    iv = F.broadcast(load_vehicle_intervals())
    cond = ((F.col("_veh") == F.col("iv_vehicle")) &
            (F.col("iv_from").isNull() | (F.col("iv_from") <= F.col("_ev_date"))) &
            (F.col("iv_to").isNull() | (F.col("_ev_date") <= F.col("iv_to"))))
    active = g.join(iv, cond, "left_semi").drop("_veh", "_ev_date")
    inactive = (g.join(iv, cond, "left_anti").drop("_veh", "_ev_date")
                 .withColumn("v", F.struct(
                     F.lit(False).alias("ok"),
                     F.lit("UNKNOWN_VEHICLE").alias("error_code"),
                     F.lit("Vehicle not active at event time").alias("error_message"),
                     F.lit(True).alias("retryable"),
                     F.col("v.flags").alias("flags"),
                     F.col("v.record_json").alias("record_json"))))
    return active, inactive


def validate(signal, lower, upper, lookups):
    """Read the window from Bronze, validate every row, split into good and bad."""
    udf = make_validate_udf(*lookups, now=simulated_now(upper))
    df = (spark.table(f"bronze.bronze_{signal}_raw")
               .where((F.col("ingestion_time") > F.lit(lower)) &
                      (F.col("ingestion_time") <= F.lit(upper))))
    validated = df.withColumn("v", udf("raw_value")).cache()
    n_bronze = validated.count()
    good = validated.where("v.ok = true")
    bad = validated.where("v.ok = false")

    # Extra check: the vehicle must be active on the event date (ref_vehicles_raw active_from / active_to)
    if ENABLE_ACTIVE_CHECK:
        good, inactive = split_by_vehicle_activity(good)
        bad = bad.unionByName(inactive)
    return validated, good, bad, n_bronze

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 8, Finished, Available, Finished, False)

## Dead-letter queue
Writes rejected rows with error code, raw text and Bronze arrival time. Idempotent: deletes the DLQ rows of the same signal and window first, so a re-run never doubles them.

In [ ]:
def write_dlq(bad, signal, batch_id, lower, upper):
    """Write rejected rows to ops.dlq_events. Idempotent: clears the same window first."""
    spark.sql(f"""DELETE FROM ops.dlq_events
                  WHERE signal='{signal}' AND stage='BATCH'
                    AND bronze_ingestion_time > TIMESTAMP'{lower:{TS_FMT}}'
                    AND bronze_ingestion_time <= TIMESTAMP'{upper:{TS_FMT}}'""")

    dlq = bad.select(
        F.expr("uuid()").alias("dlq_id"),
        F.lit(batch_id).alias("batch_id"),
        F.lit("BATCH").alias("stage"),
        F.lit(signal).alias("signal"),
        F.col("v.error_code").alias("error_code"),
        F.col("v.error_message").alias("error_message"),
        # get_json_object does not fail on broken JSON (returns NULL instead)
        F.get_json_object("raw_value", "$.event_id").alias("event_id"),
        "raw_value", "source_type", "source_file", "file_line",
        F.col("ingestion_time").alias("bronze_ingestion_time"),
        F.col("v.retryable").alias("is_retryable"),
        F.current_timestamp().alias("created_at"),
    )
    n_dlq = dlq.count()
    dlq.write.mode("append").saveAsTable("ops.dlq_events")
    return n_dlq

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 9, Finished, Available, Finished, False)

## Parse and enrich
Turns the validated JSON into typed columns, stores `dq_flags` as comma-separated text (NULL when no flags), and adds route, driver and cargo from a broadcast join with `ref_trips`.

**`lateness_sec`:** `expected_arrival = wall_start + (event_time - sim_start) / A`, then `lateness_sec = ingestion_time - expected_arrival`, in seconds. `event_time` is simulated time and `ingestion_time` is wall-clock time, so the simulated time is converted first. `ingestion_time` is the KQL ingestion time, so the value is approximate.

In [ ]:
def parse_and_enrich(good, signal):
    """Turn the cleaned JSON into real columns and add route/driver/cargo from ref_trips."""
    # Column list comes from the library, so it works for every signal
    common = ("schema_version INT, event_id STRING, vehicle_id STRING, trip_id STRING, "
              "device_id STRING, event_time STRING, sequence_number BIGINT")
    sig_cols = ", ".join(f"{n} {t}" for n, t in fs.SIGNAL_FIELDS[signal])
    rec_ddl = f"{common}, {sig_cols}"

    parsed = (good
        .withColumn("r", F.from_json("v.record_json", rec_ddl))
        .select("r.*",
        F.when(F.size("v.flags") > 0, F.concat_ws(",", "v.flags")).alias("dq_flags"),
        "ingestion_time")
        .withColumn("event_time", F.to_timestamp("event_time"))
        .withColumn("event_date", F.to_date("event_time")))        # partition column

    # ref_trips is small, so broadcast it
    trips = spark.table("ref.ref_trips").select("trip_id", "route_id", "driver_id", "cargo_type_id")
    enriched = parsed.join(F.broadcast(trips), "trip_id", "left")

    # lateness_sec = ingestion_time - expected_arrival (both as epoch seconds), where
    # expected_arrival = wall_start + (event_time - sim_start) / A   (values from run_manifest.json)
    expected_arrival = (F.lit(MANIFEST["wall_start"]) +
                        (F.col("event_time").cast("double") - F.lit(MANIFEST["sim_start"])) / F.lit(MANIFEST["A"]))
    return enriched.withColumn("lateness_sec",
                               F.round(F.col("ingestion_time").cast("double") - expected_arrival, 3))

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 10, Finished, Available, Finished, False)

## Deduplication
Keeps the first copy per `event_id`. Near-duplicates (same vehicle, signal and event time with a new `event_id`) are not removed yet. To be replaced by the library rule.

In [ ]:
def dedup_batch(enriched):
    """Keep one row per event_id (fs.DEDUP_KEY). TEMPORARY rule: first copy that arrived."""
    w = Window.partitionBy("event_id").orderBy(F.col("ingestion_time").asc())
    deduped = (enriched.withColumn("rn", F.row_number().over(w))
                       .where("rn = 1").drop("rn"))
    n_ok = enriched.count()
    n_kept = deduped.count()
    return deduped, n_kept, n_ok - n_kept     # (dataframe, rows kept, duplicates removed)

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 11, Finished, Available, Finished, False)

## Sequence flags (SPIKE, FLATLINE, IMPOSSIBLE_JUMP)
These flags need the previous row of the same vehicle, so they are computed on the deduplicated batch with window functions and appended to `dq_flags`.

- **temperature:** `SPIKE` when the value differs from the previous reading of the same vehicle and sensor by more than `SPIKE_DELTA_C`; `FLATLINE` when the last `FLATLINE_N` readings are identical.
- **gps:** `IMPOSSIBLE_JUMP` when the distance to the previous point divided by the time gap exceeds `MAX_JUMP_SPEED_KMH` (haversine distance).
- **engine, fuel:** no sequence flags.

**Limits:** the previous row is looked up inside the current batch only, so the first row of each vehicle in a batch is never flagged. The point after a jump is compared with the jump point, so it can be flagged too.

In [ ]:
def _append_flag(col, cond, flag):
    """Add a flag to the comma separated dq_flags text (NULL stays NULL when the condition is false)."""
    return F.when(cond, F.concat_ws(",", F.col(col), F.lit(flag))).otherwise(F.col(col))


def add_sequence_flags(df, signal):
    """Flags that need the previous row of the same vehicle (computed inside the batch only).
    NULL dq_flags still means "no flag raised", and a batch's first row per vehicle has no previous row."""
    if signal == "temperature":
        w = Window.partitionBy("vehicle_id", "sensor_id").orderBy("event_time", "sequence_number")
        w_n = w.rowsBetween(-(FLATLINE_N - 1), 0)
        t = F.col("temperature_value")
        spike = F.abs(t - F.lag("temperature_value").over(w)) > F.lit(SPIKE_DELTA_C)
        flat = ((F.count(t).over(w_n) == F.lit(FLATLINE_N)) &
                (F.min(t).over(w_n) == F.max(t).over(w_n)))
        return (df.withColumn("_spike", spike).withColumn("_flat", flat)
                  .withColumn("dq_flags", _append_flag("dq_flags", F.col("_spike"), "SPIKE"))
                  .withColumn("dq_flags", _append_flag("dq_flags", F.col("_flat"), "FLATLINE"))
                  .drop("_spike", "_flat"))

    if signal == "gps":
        w = Window.partitionBy("vehicle_id").orderBy("event_time", "sequence_number")
        lat1, lon1 = F.radians(F.lag("latitude").over(w)), F.radians(F.lag("longitude").over(w))
        lat2, lon2 = F.radians(F.col("latitude")), F.radians(F.col("longitude"))
        a = (F.pow(F.sin((lat2 - lat1) / 2), 2) +
             F.cos(lat1) * F.cos(lat2) * F.pow(F.sin((lon2 - lon1) / 2), 2))
        dist_km = 2 * 6371.0 * F.asin(F.sqrt(a))
        dt_h = (F.col("event_time").cast("double") -
                F.lag("event_time").over(w).cast("double")) / 3600.0
        jump = (dt_h > 0) & ((dist_km / dt_h) > F.lit(MAX_JUMP_SPEED_KMH))
        return (df.withColumn("_jump", jump)
                  .withColumn("dq_flags", _append_flag("dq_flags", F.col("_jump"), "IMPOSSIBLE_JUMP"))
                  .drop("_jump"))

    return df      # engine and fuel have no sequence flags

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 12, Finished, Available, Finished, False)

## MERGE into Silver
Inserts only `event_id`s not yet in Silver, so a re-run adds nothing.

In [ ]:
def merge_silver(deduped, signal):
    """Insert only event_ids that are not in Silver yet. Re-runs add nothing."""
    silver_cols = spark.table(f"silver.silver_{signal}").columns   # column order from the table
    deduped.select(*silver_cols).createOrReplaceTempView("batch_silver")
    spark.sql(f"""MERGE INTO silver.silver_{signal} t
                  USING batch_silver s ON t.event_id = s.event_id
                  WHEN NOT MATCHED THEN INSERT *""")

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 13, Finished, Available, Finished, False)

## Logging and watermark
One row per signal per run in `ops.ops_ingestion_log` (Bronze, Silver, DLQ and duplicate counts, status). The watermark moves only after a successful run.

In [ ]:
def write_log(batch_id, signal, lower, upper, n_bronze, n_silver, n_dlq, n_dups,
              status, started, error=None):
    """One row in ops.ops_ingestion_log per signal per run."""
    row = [(str(uuid.uuid4()), batch_id, JOB_NAME, signal, "BATCH", None,
            f"{lower:{WM_FMT}}", f"{upper:{WM_FMT}}",
            n_bronze, n_silver, n_dlq, n_dups, status,
            (error or "")[:1000] or None, started, datetime.utcnow())]
    ddl = ("log_id string, batch_id string, job_name string, signal string, stage string, "
           "source_file string, window_start_utc string, window_end_utc string, "
           "rows_bronze long, rows_silver long, rows_dlq long, rows_duplicates_removed long, "
           "status string, error_message string, started_at timestamp, finished_at timestamp")
    spark.createDataFrame(row, ddl).write.mode("append").saveAsTable("ops.ops_ingestion_log")


def set_watermark(signal, upper):
    """Save the new watermark. Must be the LAST step of a successful run."""
    spark.sql(f"""UPDATE ops.ops_watermark
                  SET watermark_utc = '{upper:{WM_FMT}}', updated_at = current_timestamp()
                  WHERE signal = '{signal}'""")

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 14, Finished, Available, Finished, False)

## Orchestration for one signal
Window, validate, DLQ, enrich, dedup, sequence flags, MERGE, log, watermark. On failure it logs FAILED and leaves the watermark unchanged so the run can be repeated. Check: bronze = silver + dlq + duplicates.

In [ ]:
def process_signal(signal):
    """Full batch job 1 for one signal: validate -> DLQ / Silver -> log -> watermark."""
    batch_id = str(uuid.uuid4())
    started = datetime.utcnow()
    lower, upper = get_window(signal)
    print(f"[{signal}] window: {lower} -> {upper}")

    if upper is None or lower >= upper:
        print(f"[{signal}] nothing to do")
        return None

    n_bronze = n_dlq = n_silver = n_dups = 0
    validated = None

    lookups = load_lookups()
    
    try:
        validated, good, bad, n_bronze = validate(
        signal, lower, upper, lookups
        )
        n_dlq = write_dlq(bad, signal, batch_id, lower, upper)
        enriched = parse_and_enrich(good, signal)
        deduped, n_silver, n_dups = dedup_batch(enriched)
        merge_silver(deduped, signal)
    except Exception as e:
        # Failed run: log it, do NOT move the watermark, so the run can be repeated
        write_log(batch_id, signal, lower, upper, n_bronze, n_silver, n_dlq, n_dups,
                  "FAILED", started, str(e))
        raise
    finally:
        if validated is not None:
            validated.unpersist()

    write_log(batch_id, signal, lower, upper, n_bronze, n_silver, n_dlq, n_dups,
              "SUCCESS", started)
    set_watermark(signal, upper)       # last step

    balanced = (n_bronze == n_silver + n_dlq + n_dups)
    print(f"[{signal}] bronze={n_bronze} silver={n_silver} dlq={n_dlq} dups={n_dups} | balanced={balanced}")
    return dict(signal=signal, bronze=n_bronze, silver=n_silver, dlq=n_dlq, dups=n_dups)

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 15, Finished, Available, Finished, False)

## Run batch job 1 for all signals, then Gate 1
`main()` calls `process_signal` for gps, engine, fuel and temperature, prints a balance table (bronze = silver + dlq + duplicates removed), then runs Gate 1.

**Gate 1:** invalid share = DLQ rows / Bronze rows, overall and per signal. The status is FAIL when the overall share or any signal is above `GATE_MAX_INVALID_PCT` (5 percent). Silver is still written when the gate fails. The status JSON is written to `ops.ops_ingestion_log` (stage `GATE_1`, count columns NULL so sums over stage `BATCH` are not doubled) and returned by the exit cell, for example `{"status": "PASS", "invalid_pct": 1.0, ...}`.

A second run with no new Bronze data prints "nothing to do" for each signal and returns PASS with the note "no new data".

In [ ]:
def run_gate1(results, max_pct=GATE_MAX_INVALID_PCT):
    """Gate 1: invalid share = DLQ rows / Bronze rows, overall and per signal. FAIL above max_pct."""
    if not results:
        return {"status": "PASS", "invalid_pct": 0.0, "threshold_pct": max_pct,
                "per_signal": {}, "failed_signals": [], "note": "no new data"}
    per = {r["signal"]: round(100.0 * r["dlq"] / r["bronze"], 3) if r["bronze"] else 0.0 for r in results}
    tot_b = sum(r["bronze"] for r in results)
    tot_d = sum(r["dlq"] for r in results)
    overall = round(100.0 * tot_d / tot_b, 3) if tot_b else 0.0
    failed = [s for s, p in per.items() if p > max_pct]
    status = "FAIL" if (overall > max_pct or failed) else "PASS"
    return {"status": status, "invalid_pct": overall, "threshold_pct": max_pct,
            "per_signal": per, "failed_signals": failed,
            "rows_bronze": tot_b, "rows_dlq": tot_d}


def write_gate_log(gate, started):
    """Same status JSON in ops.ops_ingestion_log (stage GATE_1). Count columns stay NULL so that
    sums over stage = 'BATCH' are not doubled."""
    now = datetime.now(timezone.utc).replace(tzinfo=None)
    row = [(str(uuid.uuid4()), None, JOB_NAME, "ALL", "GATE_1", None, None, None,
            None, None, None, None, gate["status"], json.dumps(gate)[:1000], started, now)]
    ddl = ("log_id string, batch_id string, job_name string, signal string, stage string, "
           "source_file string, window_start_utc string, window_end_utc string, "
           "rows_bronze long, rows_silver long, rows_dlq long, rows_duplicates_removed long, "
           "status string, error_message string, started_at timestamp, finished_at timestamp")
    spark.createDataFrame(row, ddl).write.mode("append").saveAsTable("ops.ops_ingestion_log")


def main(signals=("gps", "engine", "fuel", "temperature")):
    """Batch job 1 for all signals, then Gate 1. Returns the gate status dict."""
    started = datetime.now(timezone.utc).replace(tzinfo=None)
    results = []
    for s in signals:
        r = process_signal(s)
        if r:
            results.append(r)

    print("\n" + "=" * 60)
    print(f"{'signal':12s} {'bronze':>8} {'silver':>8} {'dlq':>6} {'dups':>6} balanced")
    for r in results:
        ok = r["bronze"] == r["silver"] + r["dlq"] + r["dups"]
        print(f"{r['signal']:12s} {r['bronze']:>8} {r['silver']:>8} {r['dlq']:>6} {r['dups']:>6} {ok}")

    gate = run_gate1(results)
    write_gate_log(gate, started)
    print("GATE 1:", json.dumps(gate))
    return gate

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 16, Finished, Available, Finished, False)

In [ ]:
gate = main()

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 17, Finished, Available, Finished, False)

/tmp/ipykernel_6713/2124151727.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  started = datetime.utcnow()


[gps] window: 2026-10-26 04:55:08.178000 -> 2026-10-26 04:55:08.178000
[gps] nothing to do


[engine] window: 2026-10-26 04:55:08.178000 -> 2026-10-26 04:55:08.178000
[engine] nothing to do
[fuel] window: 2026-10-26 04:55:08.178000 -> 2026-10-26 04:55:08.178000
[fuel] nothing to do


[temperature] window: 2026-10-26 04:53:47.740000 -> 2026-10-26 04:53:47.740000
[temperature] nothing to do

signal         bronze   silver    dlq   dups balanced


GATE 1: {"status": "PASS", "invalid_pct": 0.0, "threshold_pct": 5.0, "per_signal": {}, "failed_signals": [], "note": "no new data"}


## Exit value
Returns the Gate 1 JSON as the notebook exit value so Airflow or a pipeline can read it. Run it last; it ends the notebook run.

In [ ]:
mssparkutils.notebook.exit(json.dumps(gate))      # Airflow / pipeline reads this value

StatementMeta(, b7362455-c515-428c-ab10-326de26e5b81, 18, Finished, Available, Finished, False)

ExitValue: {"status": "PASS", "invalid_pct": 0.0, "threshold_pct": 5.0, "per_signal": {}, "failed_signals": [], "note": "no new data"}

In [ ]:
# Release cached data and clear Spark cache
spark.catalog.clearCache()